# Session 15 Tasks — Complete Python Solutions

This notebook covers all three tasks from `Session 15 Tasks.pdf`.

**Tasks**
1. Scrape a public website: title, headings, paragraphs, and at least five links.
2. Extract image URLs and download at least ten images into a local folder.
3. Collect YouTube channel name, URL, and publicly available subscriber count from channel pages you are authorized to access, then save the data to a Pandas DataFrame and CSV.

> Run the cells from top to bottom. Website layouts and access rules can change; if a website blocks requests, try another public page or use HTML you are authorized to access. Do not bypass logins, paywalls, or access controls.


## 0. Install / import libraries

In [ ]:
# If needed, install dependencies in Jupyter:
# %pip install requests beautifulsoup4 pandas

import re
import time
from pathlib import Path
from urllib.parse import urljoin, urlparse

import pandas as pd
import requests
from bs4 import BeautifulSoup

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; Session15StudentProject/1.0)"
}

def get_soup(url, timeout=20):
    """Fetch a public webpage and return a BeautifulSoup parser."""
    response = requests.get(url, headers=HEADERS, timeout=timeout)
    response.raise_for_status()
    return BeautifulSoup(response.text, "html.parser")


## Task 1 — Scrape any public website

The example uses Python.org. The code extracts the page title, headings, paragraphs, and at least five links when the page provides that many. It also cleans whitespace and stores results in organized Python structures.

In [ ]:
# You may replace this with another public page.
TASK1_URL = "https://www.python.org/"

soup = get_soup(TASK1_URL)

page_title = soup.title.get_text(" ", strip=True) if soup.title else "No title found"

headings = []
for tag in soup.find_all(re.compile(r"^h[1-6]$")):
    text = tag.get_text(" ", strip=True)
    if text:
        headings.append({"tag": tag.name, "text": text})

paragraphs = []
for p in soup.find_all("p"):
    text = " ".join(p.get_text(" ", strip=True).split())
    if text:
        paragraphs.append(text)

links = []
seen_urls = set()
for a in soup.find_all("a", href=True):
    link_text = " ".join(a.get_text(" ", strip=True).split())
    absolute_url = urljoin(TASK1_URL, a["href"].strip())
    if absolute_url.startswith(("http://", "https://")) and absolute_url not in seen_urls:
        seen_urls.add(absolute_url)
        links.append({"text": link_text or "(no link text)", "url": absolute_url})

print("PAGE TITLE")
print(page_title)

print(f"\nHEADINGS ({len(headings)})")
for item in headings:
    print(f"[{item['tag']}] {item['text']}")

print(f"\nPARAGRAPHS ({len(paragraphs)})")
for i, paragraph in enumerate(paragraphs[:10], start=1):
    print(f"{i}. {paragraph}")

print(f"\nLINKS ({len(links)} total; showing first 10)")
for item in links[:10]:
    print(f"- {item['text']}: {item['url']}")

if len(links) < 5:
    print("\nNote: this page returned fewer than five unique HTTP(S) links. Try a different public page.")


In [ ]:
# Optional: save Task 1 output to JSON for submission.
import json

task1_results = {
    "url": TASK1_URL,
    "title": page_title,
    "headings": headings,
    "paragraphs": paragraphs,
    "links": links,
}
with open("task1_results.json", "w", encoding="utf-8") as f:
    json.dump(task1_results, f, ensure_ascii=False, indent=2)

print("Saved task1_results.json")


## Task 2 — Extract and download images

This cell converts relative image paths to absolute URLs, removes duplicates, and attempts to download up to 10 images. The default page is Wikimedia Commons' public main page. Downloading may be limited by the site, network, or image formats; the code reports successes and failures.

In [ ]:
# Change this to another public webpage containing images if needed.
TASK2_URL = "https://commons.wikimedia.org/wiki/Main_Page"
IMAGE_DIR = Path("downloaded_images")
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

image_soup = get_soup(TASK2_URL)
image_urls = []
seen_images = set()

for img in image_soup.find_all("img"):
    src = (img.get("src") or img.get("data-src") or "").strip()
    if not src:
        # Some pages use srcset instead of src.
        srcset = (img.get("srcset") or "").strip()
        if srcset:
            src = srcset.split(",")[0].strip().split(" ")[0]
    if not src:
        continue

    absolute_image_url = urljoin(TASK2_URL, src)
    if absolute_image_url.startswith(("http://", "https://")) and absolute_image_url not in seen_images:
        seen_images.add(absolute_image_url)
        image_urls.append(absolute_image_url)

print(f"Found {len(image_urls)} unique image URLs.")
for url in image_urls[:15]:
    print(url)

def safe_filename(url, index):
    """Build a safe filename from the URL path, keeping an image extension if present."""
    path_name = Path(urlparse(url).path).name
    path_name = re.sub(r"[^A-Za-z0-9._-]+", "_", path_name).strip("._")
    if not path_name or "." not in path_name:
        path_name = f"image_{index:02d}.img"
    return f"{index:02d}_{path_name[:100]}"

downloaded = []
failed = []

for index, image_url in enumerate(image_urls, start=1):
    if len(downloaded) >= 10:
        break
    try:
        response = requests.get(image_url, headers=HEADERS, timeout=25, stream=True)
        response.raise_for_status()
        content_type = response.headers.get("Content-Type", "").lower()
        if not content_type.startswith("image/"):
            failed.append((image_url, f"Not an image response ({content_type or 'unknown content type'})"))
            response.close()
            continue

        filename = safe_filename(image_url, index)
        file_path = IMAGE_DIR / filename
        with open(file_path, "wb") as file:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    file.write(chunk)
        response.close()

        if file_path.stat().st_size == 0:
            file_path.unlink(missing_ok=True)
            failed.append((image_url, "Empty file"))
            continue
        downloaded.append({"url": image_url, "file": str(file_path), "bytes": file_path.stat().st_size})
        print(f"Downloaded: {file_path}")
        time.sleep(0.2)  # Be polite to the host.
    except Exception as exc:
        failed.append((image_url, str(exc)))

print(f"\nSuccessfully downloaded {len(downloaded)} image(s).")
print(f"Failed/skipped: {len(failed)}")
for url, reason in failed[:5]:
    print(f"- {url} -> {reason}")

if len(downloaded) < 10:
    print("Fewer than 10 images were downloaded. Try another public page with more images or rerun later.")


In [ ]:
# Save the image URL inventory and download report as CSV.
pd.DataFrame({"image_url": image_urls}).to_csv("task2_image_urls.csv", index=False, encoding="utf-8")
pd.DataFrame(downloaded).to_csv("task2_downloaded_images.csv", index=False, encoding="utf-8")
print("Saved task2_image_urls.csv and task2_downloaded_images.csv")


## Task 3 — YouTube channel information

For this task, enter public channel URLs you are allowed to access. The parser checks common metadata fields for the channel name and subscriber count. YouTube changes its HTML frequently, and subscriber counts may be hidden or unavailable; missing values are recorded as `Not publicly available` rather than guessed. This is ordinary page parsing, not a method to bypass access restrictions.

In [ ]:
# Replace these examples with the public YouTube channel URLs you want to analyze.
CHANNEL_URLS = [
    "https://www.youtube.com/@YouTube",
    "https://www.youtube.com/@TED",
]

def first_meta_content(soup, selectors):
    for selector in selectors:
        tag = soup.select_one(selector)
        if tag:
            value = tag.get("content") or tag.get_text(" ", strip=True)
            if value:
                return value.strip()
    return ""

def scrape_youtube_channel(channel_url):
    result = {
        "channel_name": "",
        "channel_url": channel_url,
        "subscriber_count": "Not publicly available",
    }
    try:
        soup = get_soup(channel_url)
        result["channel_name"] = (
            first_meta_content(soup, [
                'meta[property="og:title"]',
                'meta[name="title"]',
            ])
            or (soup.title.get_text(" ", strip=True) if soup.title else "")
        )

        # Try common subscriber-count metadata/text patterns.
        subscriber = first_meta_content(soup, [
            'meta[itemprop="subscriberCount"]',
            'meta[ name="description" ]',
        ])
        if subscriber:
            match = re.search(r"([\d.,]+\s*(?:K|M|B)?\s* subscribers?)", subscriber, re.I)
            if match:
                result["subscriber_count"] = match.group(1).strip()

        if result["subscriber_count"] == "Not publicly available":
            page_text = soup.get_text(" ", strip=True)
            match = re.search(r"([\d.,]+\s*(?:K|M|B)?\s* subscribers?)", page_text, re.I)
            if match:
                result["subscriber_count"] = match.group(1).strip()

        if not result["channel_name"]:
            result["channel_name"] = "Name not found"
    except Exception as exc:
        result["channel_name"] = "Could not retrieve channel page"
        result["error"] = str(exc)

    return result

channel_records = []
for channel_url in CHANNEL_URLS:
    record = scrape_youtube_channel(channel_url)
    channel_records.append(record)
    time.sleep(1)

youtube_df = pd.DataFrame(channel_records, columns=[
    "channel_name", "channel_url", "subscriber_count"
])
display(youtube_df)


In [ ]:
# Export the collected information to CSV.
youtube_df.to_csv("youtube_channel_info.csv", index=False, encoding="utf-8-sig")
print("Saved youtube_channel_info.csv")


## Submission checklist

For each task, the PDF asks for Python source code, `requirements.txt`, screenshots of the output, and a short `README.md`.

- The notebook contains the source code and can be exported to `.py` files from Jupyter if required.
- `requirements.txt` and `README.md` are included alongside this notebook in the solution package.
- Run the notebook on your computer and take screenshots of the visible outputs for your submission. I have not included fabricated screenshots; results depend on live website access.
